In [1]:
!pip install transformers sentence-transformers faiss-cpu torch sentencepiece -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 39.8 MB/s eta 0:00:00


In [2]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

from transformers import pipeline
# 1. Create the knowledge base

documents = [
"""
Generative Artificial Intelligence is a branch of AI that creates
new content such as text, images, audio, video and computer programs.
""",
"""
ware transformer-based models trained on massive
text datasets. They are used for text generation, summarization,
translation, question answering and conversational AI.
""",
"""
Retrieval-Augmented Generation combines information retrieval with
text generation. It retrieves relevant documents from an external
knowledge base and gives them to a language model as context.
""",
"""
Vector databases store high-dimensional embeddings and perform
similarity searches. Examples of vector databases include FAISS,
ChromaDB, Pinecone, Weaviate and Milvus.
""",
"""
Prompt engineering is the process of designing clear instructions
that guide a language model to produce accurate and useful responses.
Common techniques include zero-shot, few-shot and role-based prompting.
""",
"""
Fine-tuning adapts a pretrained language model to a specific domain
or task by training it further using a smaller domain-specific dataset.
"""
]

In [3]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
document_embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True
)
# Convert to float32 because FAISS requires float32 vectors
document_embeddings = document_embeddings.astype("float32")

In [5]:
faiss.normalize_L2(document_embeddings)

In [6]:
embedding_dimension = document_embeddings.shape[1]
# Inner-product search on normalized vectors gives cosine similarity
vector_database = faiss.IndexFlatIP(embedding_dimension)
# Store document vectors in the database
vector_database.add(document_embeddings)

In [8]:
generator = pipeline(
    task="text-generation",
    model="google/flan-t5-base"
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCaus

In [11]:
def retrieve_documents(query, top_k=2):
  query_embedding = embedding_model.encode(
      [query],
      convert_to_numpy=True
  ).astype("float32")
  faiss.normalize_L2(query_embedding)
  similarity_scores, document_indices = vector_database.search(
      query_embedding,
      top_k
   )
  retrieved_documents = []
  for index, score in zip(document_indices[0],similarity_scores[0]):
    retrieved_documents.append({
        "document": documents[index].strip(),
        "score": float(score)
    })
  return retrieved_documents

In [12]:
def generate_answer(query, retrieved_documents):
  context = "\n\n".join(
      item["document"] for item in retrieved_documents
    )
  prompt = f"""
  Answer the question using only the information provided in the context.
  Context:
  {context}
  Question:
  {query}
  Instructions:
  1. Give a clear and concise answer.
  2. Do not add information that is not present in the context.
  3. If the answer is unavailable, state:
  "The answer is not available in the knowledge base."
  Answer:
  """
  result = generator(
      prompt,
      max_new_tokens=150,
      do_sample=False
  )
  return result[0]["generated_text"]

In [17]:
print("RETRIEVAL-AUGMENTED GENERATION SYSTEM")
print("=" * 55)
user_query = input("\nEnter your question: ")
retrieved_results = retrieve_documents(
    query=user_query,
    top_k=2
)
answer = generate_answer(
    query=user_query,
    retrieved_documents=retrieved_results
)
print("\nRETRIEVED DOCUMENTS")
print("-" * 55)
for number, item in enumerate(retrieved_results, start=1):
  print(f"\nDocument {number}:")
  print(item["document"])
  print(f"Similarity Score: {item['score']:.4f}")
print("\nGENERATED ANSWER")
print("-" * 55)
print(answer)

RETRIEVAL-AUGMENTED GENERATION SYSTEM

Enter your question: What is Retrieval-Augmented Generation?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



RETRIEVED DOCUMENTS
-------------------------------------------------------

Document 1:
Retrieval-Augmented Generation combines information retrieval with
text generation. It retrieves relevant documents from an external
knowledge base and gives them to a language model as context.
Similarity Score: 0.6933

Document 2:
Generative Artificial Intelligence is a branch of AI that creates
new content such as text, images, audio, video and computer programs.
Similarity Score: 0.3435

GENERATED ANSWER
-------------------------------------------------------

  Answer the question using only the information provided in the context.
  Context:
  Retrieval-Augmented Generation combines information retrieval with
text generation. It retrieves relevant documents from an external
knowledge base and gives them to a language model as context.

Generative Artificial Intelligence is a branch of AI that creates
new content such as text, images, audio, video and computer programs.
  Question:
  What is 